# moveEnetOFK — Hyperparameter Optimisation (NNI)

Build order:
- **Block 1 — Sanity check** ← you are here
- Block 2 — Paths & config
- Block 3 — Dataset picker
- Block 4 — Trial logic
- Block 5 — Sync → `trial.py`
- Block 6 — Launch NNI
- Block 7 — Monitor
- Block 8 — Results / best params

In [7]:
## ─── BLOCK 1 · SANITY CHECK ──────────────────────────────────────────────────
import sys, subprocess
from pathlib import Path

BINARY    = Path("/home/moveEnetFlow/build/moveEnetOFK_offline")
DATA_ROOT = Path("/data/moveEnet_test/raw")
HPE_CORE  = Path("/usr/local/src/hpe-core")

ok = True
def chk(label, cond, detail=""):
    global ok
    sym = "✓" if cond else "✗"
    if not cond:
        ok = False
    print(f"  {sym}  {label:<40} {detail}")

print("── Python environment ──────────────────────────────")
chk("Python executable", True, sys.executable)
chk("Python version", sys.version_info >= (3, 8), sys.version.split()[0])

print("\n── Core Python packages ────────────────────────────")
for pkg in ["nni", "numpy", "scipy", "matplotlib", "pandas", "yaml"]:
    try:
        m = __import__(pkg)
        ver = getattr(m, "__version__", "?")
        chk(pkg, True, ver)
    except ImportError as e:
        chk(pkg, False, str(e))

print("\n── CLI tools ───────────────────────────────────────")
r = subprocess.run(["nnictl", "--version"], capture_output=True, text=True)
chk("nnictl", r.returncode == 0, r.stdout.strip() or r.stderr.strip())

print("\n── File system ─────────────────────────────────────")
chk("C++ binary", BINARY.exists(), str(BINARY))
chk("Data root", DATA_ROOT.exists(), str(DATA_ROOT))
chk("hpe-core root", HPE_CORE.exists(), str(HPE_CORE))

print("\n── hpe-core imports (subprocess, avoids urllib3 conflict) ──")
probe = subprocess.run(
    [sys.executable, "-c",
     f"import sys; sys.path.insert(0,'{HPE_CORE}');"
     "from datasets.utils import constants, parsing;"
     "from evaluation.utils import metrics;"
     "m = metrics.MPJPE(); print('OK')"],
    capture_output=True, text=True
)
chk("datasets.utils.constants / parsing", "OK" in probe.stdout, probe.stderr.strip()[:80] or "OK")
chk("evaluation.utils.metrics (MPJPE)", "OK" in probe.stdout, "")

print("\n── Datasets ────────────────────────────────────────")
if DATA_ROOT.exists():
    ds_list = sorted(d.name for d in DATA_ROOT.iterdir()
                     if d.is_dir() and (d / "ch0dvs" / "data.log").exists())
    chk("datasets with event log", len(ds_list) > 0, f"{len(ds_list)} found")
    for ds in ds_list[:3]:
        print(f"       {ds}")
    if len(ds_list) > 3:
        print(f"       ... and {len(ds_list)-3} more")

print()
print("══ Overall:", "ALL OK ✓" if ok else "ISSUES FOUND ✗ — fix before proceeding")

── Python environment ──────────────────────────────
  ✓  Python executable                        /bin/python3
  ✓  Python version                           3.8.10

── Core Python packages ────────────────────────────
  ✓  nni                                      3.0
  ✓  numpy                                    1.24.4
  ✓  scipy                                    1.10.1
  ✓  matplotlib                               3.7.5
  ✓  pandas                                   2.0.3
  ✓  yaml                                     6.0.3

── CLI tools ───────────────────────────────────────
  ✓  nnictl                                   3.0

── File system ─────────────────────────────────────
  ✓  C++ binary                               /home/moveEnetFlow/build/moveEnetOFK_offline
  ✓  Data root                                /data/moveEnet_test/raw
  ✓  hpe-core root                            /usr/local/src/hpe-core

── hpe-core imports (subprocess, avoids urllib3 conflict) ──
  ✓  datasets.util

In [9]:
## ─── BLOCK 2 · PATHS & CONFIG ────────────────────────────────────────────────
# Edit this cell to change what NNI explores.
# All later blocks read from these variables — only edit here.

from pathlib import Path

# ── Fixed paths ────────────────────────────────────────────────────────────────
BINARY      = Path("/home/moveEnetFlow/build/moveEnetOFK_offline")
DATA_ROOT   = Path("/data/moveEnet_test/raw")
HPE_CORE    = Path("/usr/local/src/hpe-core")
TRIAL_PY    = Path("/home/moveEnetFlow/utils/trial.py")
CSV_DIR     = Path("/tmp/nni_hpo"); CSV_DIR.mkdir(exist_ok=True)

# ── NNI experiment settings ────────────────────────────────────────────────────
TUNER         = "TPE"       # TPE | Random | Anneal | Evolution
OPTIMIZE_MODE = "minimize"  # we minimise MPJPE
MAX_TRIALS    = 2          # total parameter sets to try
CONCURRENCY   = 1           # keep at 1: each trial monopolises YARP/MoveNet
NNI_PORT      = 8081        # web-portal port

# ── Search space ───────────────────────────────────────────────────────────────
SEARCH_SPACE = {
    "pu":          {"_type": "loguniform", "_value": [1e-3, 1.0  ]},
    "muD":         {"_type": "loguniform", "_value": [1e-6, 1e-2 ]},
    "muV":         {"_type": "loguniform", "_value": [1e-6, 1e-2 ]},
    "flow_period": {"_type": "choice",     "_value": [0.001, 0.005, 0.01]},
    "net_period":  {"_type": "choice",     "_value": [0.005, 0.01, 0.05, 0.1]},
}

# ── Fixed params (not tuned) ───────────────────────────────────────────────────
FIXED_OUTPUT_PERIOD = 0.005   # CSV write cadence (s)
roi = 20                     # region of interest (mm) — only used if roi param is not tuned
# ── Summary ────────────────────────────────────────────────────────────────────
import json
print(f"Binary       : {BINARY}")
print(f"Data root    : {DATA_ROOT}")
print(f"trial.py     : {TRIAL_PY}")
print(f"CSV scratch  : {CSV_DIR}")
print(f"Tuner        : {TUNER}  |  max trials: {MAX_TRIALS}  |  port: {NNI_PORT}")
print(f"\nSearch space ({len(SEARCH_SPACE)} params):")
for k, v in SEARCH_SPACE.items():
    print(f"  {k:<14} {v}")

Binary       : /home/moveEnetFlow/build/moveEnetOFK_offline
Data root    : /data/moveEnet_test/raw
trial.py     : /home/moveEnetFlow/utils/trial.py
CSV scratch  : /tmp/nni_hpo
Tuner        : TPE  |  max trials: 2  |  port: 8081

Search space (5 params):
  pu             {'_type': 'loguniform', '_value': [0.001, 1.0]}
  muD            {'_type': 'loguniform', '_value': [1e-06, 0.01]}
  muV            {'_type': 'loguniform', '_value': [1e-06, 0.01]}
  flow_period    {'_type': 'choice', '_value': [0.001, 0.005, 0.01]}
  net_period     {'_type': 'choice', '_value': [0.005, 0.01, 0.05, 0.1]}


In [11]:
## ─── BLOCK 3 · DATASET PICKER ────────────────────────────────────────────────
# Auto-discovers all available sequences under DATA_ROOT.
# Edit SELECTED_DATASETS to choose which ones feed each trial.
# Shorter sequences → faster trials; mix subjects/actions for a more robust score.

# ── Known-bad sequences (corrupted GT) — always skipped ───────────────────────
EXCLUDED = {
    "cam4_S11_Greeting", "cam4_S9_Eating", "cam4_S9_SittingDown_1",
    "cam4_S9_Walking",   "cam4_S9_Waiting_1", "cam4_S9_Photo",
    "cam4_S9_Photo_1",   "cam4_S9_Greeting",
}

# ── Auto-discover ──────────────────────────────────────────────────────────────
ALL_DATASETS = sorted(
    d.name for d in DATA_ROOT.iterdir()
    if d.is_dir()
    and (d / "ch0dvs" / "data.log").exists()
    and d.name not in EXCLUDED
)

# ── Your selection — edit freely ───────────────────────────────────────────────
SELECTED_DATASETS = [
    "cam2_S1_Directions",
    "cam2_S1_Discussion",
    "cam2_S1_Eating",
    "cam2_S1_Greeting",
    "cam2_S1_Phoning",
    "cam2_S1_Posing",
    "cam2_S1_Purchases",
    "cam2_S1_Sitting_1",
    "cam2_S1_SittingDown",
    "cam2_S1_Smoking",
    "cam2_S1_TakingPhoto",
    "cam2_S1_Waiting",
    "cam2_S1_Walking",
    "cam2_S1_WalkingDog",
    "cam2_S1_WalkTogether",
]

# ── Validate ───────────────────────────────────────────────────────────────────
GT_CANDIDATES = ["ch0GT200Hzskeleton", "ch0GT50Hzskeleton"]  # checked in order

print(f"{'Dataset':<35} {'Events':^8} {'GT':^22} {'Status'}")
print("─" * 75)

valid_datasets = []
for ds_name in SELECTED_DATASETS:
    ds_path = DATA_ROOT / ds_name

    # event log
    ev_ok = (ds_path / "ch0dvs" / "data.log").exists()

    # ground-truth log (first match wins)
    gt_found = next((g for g in GT_CANDIDATES if (ds_path / g / "data.log").exists()), None)

    in_all   = ds_name in ALL_DATASETS
    status   = "✓" if (ev_ok and gt_found and in_all) else "✗"
    gt_label = gt_found if gt_found else "MISSING"

    print(f"  {status}  {ds_name:<33} {'✓' if ev_ok else '✗':^8} {gt_label:<22}")

    if status == "✓":
        valid_datasets.append(ds_name)

print()
print(f"Selected: {len(SELECTED_DATASETS)}  |  Valid (events + GT): {len(valid_datasets)}  |  Available total: {len(ALL_DATASETS)}")

if len(valid_datasets) == 0:
    raise RuntimeError("No valid datasets found — check DATA_ROOT and GT folders.")
if len(valid_datasets) < len(SELECTED_DATASETS):
    print(f"\n⚠  {len(SELECTED_DATASETS) - len(valid_datasets)} dataset(s) will be skipped (missing events or GT).")

# Override with only the validated subset for later blocks
SELECTED_DATASETS = valid_datasets
print(f"\nSELECTED_DATASETS updated → {len(SELECTED_DATASETS)} sequences will be used per trial.")

Dataset                              Events            GT           Status
───────────────────────────────────────────────────────────────────────────
  ✓  cam2_S1_Directions                   ✓     ch0GT200Hzskeleton    
  ✓  cam2_S1_Discussion                   ✓     ch0GT200Hzskeleton    
  ✓  cam2_S1_Eating                       ✓     ch0GT200Hzskeleton    
  ✓  cam2_S1_Greeting                     ✓     ch0GT200Hzskeleton    
  ✓  cam2_S1_Phoning                      ✓     ch0GT200Hzskeleton    
  ✓  cam2_S1_Posing                       ✓     ch0GT200Hzskeleton    
  ✓  cam2_S1_Purchases                    ✓     ch0GT200Hzskeleton    
  ✓  cam2_S1_Sitting_1                    ✓     ch0GT200Hzskeleton    
  ✓  cam2_S1_SittingDown                  ✓     ch0GT200Hzskeleton    
  ✓  cam2_S1_Smoking                      ✓     ch0GT200Hzskeleton    
  ✓  cam2_S1_TakingPhoto                  ✓     ch0GT200Hzskeleton    
  ✓  cam2_S1_Waiting                      ✓     ch0GT200Hzskeleton  

In [12]:
## ─── BLOCK 4 · TRIAL LOGIC ───────────────────────────────────────────────────
# Defines two standalone functions that Block 5 will extract verbatim into trial.py.
# ─ All external dependencies are passed as arguments so inspect.getsource() yields
#   a fully self-contained script with zero hidden imports.
# ─ Edit freely; re-run Block 5 to push changes to trial.py.

import inspect  # needed here so Block 5 can call inspect.getsource()

# ── joint order must match KEYPOINTS_MAP ──────────────────────────────────────
_JOINT_KEYS = [
    "head", "shoulder_right", "shoulder_left",
    "elbow_right", "elbow_left",
    "hip_left", "hip_right",
    "wrist_right", "wrist_left",
    "knee_right", "knee_left",
    "ankle_right", "ankle_left",
]  # 13 joints → indices 0-12


def _compute_mpjpe(csv_path, dataset_name, data_root, gt_candidates,
                   parsing, metrics, interpolate, np):
    """
    Load a prediction CSV and the matching GT skeleton log, interpolate GT to
    the prediction timestamps, and return mean MPJPE in pixels.
    Returns float('nan') on any error.
    """
    # ── load predictions ──────────────────────────────────────────────────────
    try:
        raw = np.loadtxt(str(csv_path), delimiter=',', skiprows=1)
    except Exception as e:
        print(f"    CSV load error: {e}")
        return float('nan')

    if raw.ndim < 2 or len(raw) == 0:
        print("    CSV empty or 1-row")
        return float('nan')

    ts_pred     = raw[:, 0]                         # (N,)
    joints_pred = raw[:, 2:28].reshape(-1, 13, 2)   # (N, 13, 2)  cols 2..27

    # ── locate GT log ─────────────────────────────────────────────────────────
    ds_root = data_root / dataset_name
    gt_log  = next(
        (ds_root / g / "data.log" for g in gt_candidates
         if (ds_root / g / "data.log").exists()),
        None
    )
    if gt_log is None:
        print("    GT log not found")
        return float('nan')

    # ── load GT ───────────────────────────────────────────────────────────────
    try:
        data = parsing.import_yarp_skeleton_data(gt_log, multi_channel=False)
    except Exception as e:
        print(f"    GT parse error: {e}")
        return float('nan')

    ts_gt = data['ts']                              # (N_gt,)

    # ── build per-joint interpolators ─────────────────────────────────────────
    interp = {}
    for jk in _JOINT_KEYS:
        xy = data[jk]                               # (N_gt, 2)
        interp[jk] = (
            interpolate.interp1d(ts_gt, xy[:, 0], bounds_error=False, fill_value='extrapolate'),
            interpolate.interp1d(ts_gt, xy[:, 1], bounds_error=False, fill_value='extrapolate'),
        )

    # ── interpolate GT at prediction timestamps ────────────────────────────────
    ts_pred_clipped = np.clip(ts_pred, ts_gt[0], ts_gt[-1])
    joints_gt = np.zeros_like(joints_pred)           # (N, 13, 2)
    for j, jk in enumerate(_JOINT_KEYS):
        joints_gt[:, j, 0] = interp[jk][0](ts_pred_clipped)
        joints_gt[:, j, 1] = interp[jk][1](ts_pred_clipped)

    # ── sanity-check for degenerate GT ────────────────────────────────────────
    joints_gt[np.abs(joints_gt) > 1e6] = np.nan
    mask = np.isfinite(joints_gt).all(axis=(1, 2)) & np.isfinite(joints_pred).all(axis=(1, 2))
    if mask.sum() == 0:
        print("    No valid frames after NaN filter")
        return float('nan')

    # ── compute MPJPE ─────────────────────────────────────────────────────────
    m = metrics.MPJPE()
    m.update_samples(joints_pred[mask], joints_gt[mask])
    _, total_mpjpe = m.get_value()
    return float(total_mpjpe)


def run_trial(params, selected_datasets, binary, data_root, csv_dir,
              fixed_output_period, fixed_roi, gt_candidates,
              parsing, metrics, interpolate, np, subprocess, time):
    """
    Run the C++ binary once per dataset and return the mean MPJPE across all
    datasets that succeeded.  Returns 9999.0 if every dataset fails.
    """
    scores   = []
    trial_id = params.get('_trial_id', 'manual')
    roi_val  = int(params.get('roi', fixed_roi))

    for ds in selected_datasets:
        event_log = data_root / ds / "ch0dvs" / "data.log"
        if not event_log.exists():
            print(f"  [{ds}] event log missing — skip")
            continue

        csv_out = csv_dir / f"{trial_id}_{ds}.csv"
        cmd = [
            str(binary),
            "--data_file",     str(event_log),
            "--output_csv",    str(csv_out),
            "--pu",            str(params["pu"]),
            "--muD",           str(params["muD"]),
            "--muV",           str(params["muV"]),
            "--roi",           str(roi_val),
            "--flow_period",   str(params["flow_period"]),
            "--net_period",    str(params["net_period"]),
            "--output_period", str(fixed_output_period),
            "--no_video",
        ]

        print(f"  [{ds}] running ...", end=" ", flush=True)
        t0 = time.time()
        try:
            proc = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
        except subprocess.TimeoutExpired:
            print("TIMEOUT")
            continue

        elapsed = time.time() - t0
        if proc.returncode != 0:
            print(f"FAILED (rc={proc.returncode})")
            print("   STDERR:", proc.stderr[-300:])
            continue

        print(f"{elapsed:.1f}s", end=" → ")
        v = _compute_mpjpe(csv_out, ds, data_root, gt_candidates,
                           parsing, metrics, interpolate, np)
        print(f"MPJPE = {v:.2f} px" if not np.isnan(v) else "MPJPE = NaN")
        if not np.isnan(v):
            scores.append(v)

    result = float(np.mean(scores)) if scores else 9999.0
    n = len(scores)
    print(f"  ── trial {trial_id}: mean MPJPE = {result:.2f} px ({n}/{len(selected_datasets)} datasets)")
    return result


# ── dry-run check: can we call the functions without errors? ──────────────────
print("_compute_mpjpe  defined ✓")
print("run_trial        defined ✓")
print(f"Joint order ({len(_JOINT_KEYS)} joints):", _JOINT_KEYS)

_compute_mpjpe  defined ✓
run_trial        defined ✓
Joint order (13 joints): ['head', 'shoulder_right', 'shoulder_left', 'elbow_right', 'elbow_left', 'hip_left', 'hip_right', 'wrist_right', 'wrist_left', 'knee_right', 'knee_left', 'ankle_right', 'ankle_left']


In [13]:
## ─── BLOCK 5 · SYNC → trial.py ───────────────────────────────────────────────
# Extracts the two functions from Block 4 and writes a self-contained trial.py.
# Re-run this cell every time you edit Block 4.

import ast, textwrap

# ── header: imports + constants baked in from Block 2/3 ───────────────────────
_header = textwrap.dedent(f"""\
    # trial.py — auto-generated by Block 5 of moveEnetOFK_HPO.ipynb
    # DO NOT EDIT BY HAND — re-run Block 5 to regenerate.

    import sys, subprocess, time
    sys.path.insert(0, '{HPE_CORE}')

    import numpy as np
    from pathlib import Path
    from scipy import interpolate
    from datasets.utils import parsing
    from evaluation.utils import metrics
    import nni

    # ── constants ────────────────────────────────────────────────────────────────
    BINARY              = Path('{BINARY}')
    DATA_ROOT           = Path('{DATA_ROOT}')
    CSV_DIR             = Path('{CSV_DIR}'); CSV_DIR.mkdir(exist_ok=True)
    SELECTED_DATASETS   = {SELECTED_DATASETS!r}
    FIXED_OUTPUT_PERIOD = {FIXED_OUTPUT_PERIOD!r}
    FIXED_ROI           = {roi!r}
    GT_CANDIDATES       = {GT_CANDIDATES!r}

    _JOINT_KEYS = {_JOINT_KEYS!r}

""")

# ── extract function bodies from Block 4 via inspect ─────────────────────────
_body = (
    inspect.getsource(_compute_mpjpe) + "\n\n"
    + inspect.getsource(run_trial)
)

# ── __main__ entrypoint ───────────────────────────────────────────────────────
_main = textwrap.dedent("""\

    if __name__ == "__main__":
        params = nni.get_next_parameter()
        params['_trial_id'] = nni.get_trial_id()
        print(f"[NNI trial {params['_trial_id']}] params = {params}")

        result = run_trial(
            params           = params,
            selected_datasets= SELECTED_DATASETS,
            binary           = BINARY,
            data_root        = DATA_ROOT,
            csv_dir          = CSV_DIR,
            fixed_output_period = FIXED_OUTPUT_PERIOD,
            fixed_roi        = FIXED_ROI,
            gt_candidates    = GT_CANDIDATES,
            parsing          = parsing,
            metrics          = metrics,
            interpolate      = interpolate,
            np               = np,
            subprocess       = subprocess,
            time             = time,
        )

        nni.report_final_result(result)
""")

# ── write ─────────────────────────────────────────────────────────────────────
trial_src = _header + _body + _main
TRIAL_PY.write_text(trial_src)
print(f"Written : {TRIAL_PY}  ({len(trial_src)} chars)")

# ── syntax check ─────────────────────────────────────────────────────────────
try:
    ast.parse(trial_src)
    print("Syntax  : OK ✓")
except SyntaxError as e:
    print(f"Syntax  : ERROR ✗  {e}")
    raise

# ── quick preview (first 30 lines) ───────────────────────────────────────────
print("\n── trial.py preview (first 30 lines) ─────────────────────────────────")
for i, line in enumerate(trial_src.splitlines()[:30], 1):
    print(f"  {i:3}  {line}")

Written : /home/moveEnetFlow/utils/trial.py  (7168 chars)
Syntax  : OK ✓

── trial.py preview (first 30 lines) ─────────────────────────────────
    1  # trial.py — auto-generated by Block 5 of moveEnetOFK_HPO.ipynb
    2  # DO NOT EDIT BY HAND — re-run Block 5 to regenerate.
    3  
    4  import sys, subprocess, time
    5  sys.path.insert(0, '/usr/local/src/hpe-core')
    6  
    7  import numpy as np
    8  from pathlib import Path
    9  from scipy import interpolate
   10  from datasets.utils import parsing
   11  from evaluation.utils import metrics
   12  import nni
   13  
   14  # ── constants ────────────────────────────────────────────────────────────────
   15  BINARY              = Path('/home/moveEnetFlow/build/moveEnetOFK_offline')
   16  DATA_ROOT           = Path('/data/moveEnet_test/raw')
   17  CSV_DIR             = Path('/tmp/nni_hpo'); CSV_DIR.mkdir(exist_ok=True)
   18  SELECTED_DATASETS   = ['cam2_S1_Directions', 'cam2_S1_Discussion', 'cam2_S1_Eating', 'cam2_S1_

In [ ]:
## ─── BLOCK 6 · LAUNCH NNI ────────────────────────────────────────────────────
# Pre-flight checks → write config files → nnictl create.
# NNI runs as a background daemon; this cell returns immediately.
# ⚠ Prerequisites: yarpserver must be running before executing this cell.

import json, re, time as _time

NNI_EXPERIMENT_ID = None   # will be set after successful launch

# ── 1. YARP server check ──────────────────────────────────────────────────────
yarp_r = subprocess.run(["yarp", "where"], capture_output=True, text=True, timeout=5)
yarp_ok = yarp_r.returncode == 0
print(f"YARP server  : {'OK ✓' if yarp_ok else 'NOT FOUND ✗ — run: yarpserver &'}")
if not yarp_ok:
    raise RuntimeError("YARP server is not running. Start it first: yarpserver &")

# ── 2. Stop any stale NNI experiment on this port ────────────────────────────
stop_r = subprocess.run(["nnictl", "stop", "--all"], capture_output=True, text=True)
if stop_r.returncode == 0:
    print(f"NNI          : stopped previous experiment (if any)")
_time.sleep(1)

# Free the TCP port if still occupied
port_r = subprocess.run(["ss", "-lntp", f"sport = :{NNI_PORT}"],
                        capture_output=True, text=True)
if f":{NNI_PORT}" in port_r.stdout:
    subprocess.run(["fuser", "-k", f"{NNI_PORT}/tcp"], capture_output=True)
    _time.sleep(2)
    print(f"Port {NNI_PORT}     : freed ✓")
else:
    print(f"Port {NNI_PORT}     : free ✓")

# ── 3. Write search_space.json ────────────────────────────────────────────────
SS_FILE = CSV_DIR / "search_space.json"
SS_FILE.write_text(json.dumps(SEARCH_SPACE, indent=2))
print(f"Search space : {SS_FILE} ✓")

# ── 4. Write nni_config.yml ───────────────────────────────────────────────────
CFG_FILE = CSV_DIR / "nni_config.yml"
CFG_FILE.write_text(f"""\
experimentName: moveEnetOFK_HPO
trialCommand: python3 {TRIAL_PY}
trialCodeDirectory: {TRIAL_PY.parent}
searchSpaceFile: {SS_FILE}
trialConcurrency: {CONCURRENCY}
maxTrialNumber: {MAX_TRIALS}
tuner:
  name: {TUNER}
  classArgs:
    optimize_mode: {OPTIMIZE_MODE}
trainingService:
  platform: local
""")
print(f"NNI config   : {CFG_FILE} ✓")

# ── 5. Launch ─────────────────────────────────────────────────────────────────
print(f"\nLaunching NNI (tuner={TUNER}, max_trials={MAX_TRIALS}, port={NNI_PORT}) …")
launch_r = subprocess.run(
    ["nnictl", "create", "--config", str(CFG_FILE), "--port", str(NNI_PORT)],
    capture_output=True, text=True, timeout=30
)
# strip ANSI escape codes before parsing
_clean = re.sub(r'\x1b\[[0-9;]*m', '', launch_r.stdout)
print(_clean.strip())
if launch_r.returncode != 0:
    print("STDERR:", launch_r.stderr.strip())
    raise RuntimeError("nnictl create failed — see stderr above")

# ── 6. Extract experiment ID (from ANSI-stripped output) ──────────────────────
m = re.search(r"Experiment ID[^\w]*(\w+)", _clean)
NNI_EXPERIMENT_ID = m.group(1) if m else None

print(f"\n{'─'*55}")
print(f"  Experiment ID : {NNI_EXPERIMENT_ID}")
print(f"  Web portal    : http://localhost:{NNI_PORT}")
print(f"  Max trials    : {MAX_TRIALS}  |  Tuner: {TUNER}")
print(f"{'─'*55}")
print("Run Block 7 to monitor progress.")

YARP server  : OK ✓
NNI          : stopped previous experiment (if any)
Port 8081     : free ✓
Search space : /tmp/nni_hpo/search_space.json ✓
NNI config   : /tmp/nni_hpo/nni_config.yml ✓

Launching NNI (tuner=TPE, max_trials=2, port=8081) …
[2026-02-20 11:51:49] Creating experiment, Experiment ID: a9zwoenu
[2026-02-20 11:51:49] Starting web server...
[2026-02-20 11:51:50] Setting up...
[2026-02-20 11:51:50] Web portal URLs: http://127.0.0.1:8081 http://192.168.10.195:8081 http://10.240.78.55:8081 http://172.17.0.1:8081
[2026-02-20 11:51:50] To stop experiment run "nnictl stop a9zwoenu" or "nnictl stop --all"
[2026-02-20 11:51:50] Reference: https://nni.readthedocs.io/en/stable/reference/nnictl.html

───────────────────────────────────────────────────────
  Experiment ID : 36ma9zwoenu
  Web portal    : http://localhost:8081
  Max trials    : 2  |  Tuner: TPE
───────────────────────────────────────────────────────
Run Block 7 to monitor progress.


: 